# SCADA Flow — Automated Data Pipeline

**mcom technology** | © 2026 All rights reserved.

This notebook is the **Databricks-side** of the automated pipeline. It runs as a scheduled Lakeflow Job (every 24 hours) and performs:
1. **Incremental cleaning** of raw tables into `workspace.scada_cleaned`
2. **Data quality validation** with row count and freshness checks
3. **Pipeline run logging** to an audit table

**Trigger:** Called automatically after the local ingestion script pushes new data to the raw Delta tables.

In [0]:
import time
from datetime import datetime

# Pipeline run metadata
run_id = dbutils.widgets.get("run_id") if "run_id" in [w.name for w in dbutils.widgets.getAll()] else f"manual_{int(time.time())}"
run_start = datetime.now()
print(f"Pipeline run: {run_id}")
print(f"Started at: {run_start}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.scada_cleaned;
CREATE SCHEMA IF NOT EXISTS workspace.scada_pipeline;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.scada_pipeline.run_log (
  run_id STRING,
  step_name STRING,
  status STRING,
  rows_affected LONG,
  started_at TIMESTAMP,
  completed_at TIMESTAMP,
  duration_seconds DOUBLE,
  error_message STRING
)

In [0]:
def log_step(step_name, status, rows_affected=0, error_message=None):
    """Log a pipeline step result to the audit table."""
    completed = datetime.now()
    duration = (completed - run_start).total_seconds()
    err = error_message.replace("'", "''") if error_message else ""
    spark.sql(f"""
        INSERT INTO workspace.scada_pipeline.run_log
        VALUES ('{run_id}', '{step_name}', '{status}', {rows_affected},
                '{run_start}', '{completed}', {duration}, '{err}')
    """)
    print(f"  [{status}] {step_name}: {rows_affected:,} rows ({duration:.1f}s)")

print("Pipeline helper functions loaded.")

In [0]:
try:
    spark.sql("""
        CREATE OR REPLACE TABLE workspace.scada_cleaned.device_list AS
        SELECT
          TRIM(IDD) AS device_id, UID AS uid,
          TRIM(DEVIEC_NAME) AS device_name, TRIM(SHORT_NAME) AS short_name,
          TRY_CAST(INS_DATE AS DATE) AS installation_date,
          TRIM(IMEI_NUMBER) AS imei, TRIM(SIM_NUMBER) AS sim_number,
          TRY_CAST(SERVICE_DATE AS DATE) AS service_expiry_date,
          TRIM(STATUS) AS status, TRIM(OTHER_1) AS application_type,
          TRY_CAST(OTHER_2 AS DOUBLE) AS latitude,
          TRY_CAST(OTHER_3 AS DOUBLE) AS longitude,
          TRIM(OTHER_4) AS location_name, TRIM(OTHER_5) AS site_code
        FROM workspace.scada_flow_2026.tb_device_list
    """)
    cnt = spark.sql("SELECT COUNT(*) as c FROM workspace.scada_cleaned.device_list").collect()[0]["c"]
    log_step("device_list", "SUCCESS", cnt)
except Exception as e:
    log_step("device_list", "FAILED", error_message=str(e))
    raise

In [0]:
try:
    spark.sql("""
        CREATE OR REPLACE TABLE workspace.scada_cleaned.device_parameters AS
        SELECT
          TRIM(IDD) AS param_id, UID AS uid, TRIM(IMEI) AS imei,
          PINDEX AS parameter_index, TRIM(PDN) AS parameter_name,
          TRIM(SN) AS short_code, TRIM(IM) AS mapping_code,
          TRIM(AF) AS alert_flag, TRIM(UNIT) AS unit,
          TRY_CAST(LOW AS DOUBLE) AS low_threshold,
          TRY_CAST(MAX AS DOUBLE) AS high_threshold,
          TRIM(TYPE) AS param_type, TRIM(STATUS) AS status
        FROM workspace.scada_flow_2026.tb_device_list_parameter
    """)
    cnt = spark.sql("SELECT COUNT(*) as c FROM workspace.scada_cleaned.device_parameters").collect()[0]["c"]
    log_step("device_parameters", "SUCCESS", cnt)
except Exception as e:
    log_step("device_parameters", "FAILED", error_message=str(e))
    raise

In [0]:
try:
    spark.sql("""
        CREATE OR REPLACE TABLE workspace.scada_cleaned.sensor_log AS
        WITH deduped AS (
          SELECT
            TRIM(IMEI) AS imei, WAY AS data_way, DATA_STAMP AS data_timestamp,
            TRY_CAST(P1 AS DOUBLE) AS p1, TRY_CAST(P2 AS DOUBLE) AS p2,
            TRY_CAST(P3 AS DOUBLE) AS p3, TRY_CAST(P4 AS DOUBLE) AS p4,
            TRY_CAST(P5 AS DOUBLE) AS p5, TRY_CAST(P6 AS DOUBLE) AS p6,
            TRY_CAST(P7 AS DOUBLE) AS p7, TRY_CAST(P8 AS DOUBLE) AS p8,
            TRY_CAST(P9 AS DOUBLE) AS p9, TRY_CAST(P10 AS DOUBLE) AS p10,
            TRY_CAST(P11 AS DOUBLE) AS p11, TRY_CAST(P12 AS DOUBLE) AS p12,
            TRY_CAST(P13 AS DOUBLE) AS p13, TRY_CAST(P14 AS DOUBLE) AS p14,
            TRY_CAST(P15 AS DOUBLE) AS p15, TRY_CAST(P16 AS DOUBLE) AS p16,
            TRY_CAST(P17 AS DOUBLE) AS p17, TRY_CAST(P18 AS DOUBLE) AS p18,
            TRY_CAST(P19 AS DOUBLE) AS p19, TRY_CAST(P20 AS DOUBLE) AS p20,
            TRY_CAST(P21 AS DOUBLE) AS p21, TRY_CAST(P22 AS DOUBLE) AS p22,
            TRY_CAST(P23 AS DOUBLE) AS p23, TRY_CAST(P24 AS DOUBLE) AS p24,
            TRY_CAST(P25 AS DOUBLE) AS p25, TRY_CAST(P26 AS DOUBLE) AS p26,
            TRY_CAST(P27 AS DOUBLE) AS p27, TRY_CAST(P28 AS DOUBLE) AS p28,
            TRY_CAST(P29 AS DOUBLE) AS p29, TRY_CAST(P30 AS DOUBLE) AS p30,
            SERVER_DT AS server_timestamp,
            ROW_NUMBER() OVER (
              PARTITION BY TRIM(IMEI), DATA_STAMP, WAY
              ORDER BY SERVER_DT DESC
            ) AS rn
          FROM workspace.scada_flow_2025.tb_logedata
          WHERE YEAR(DATA_STAMP) BETWEEN 2020 AND 2026
        )
        SELECT * EXCEPT(rn) FROM deduped WHERE rn = 1
    """)
    cnt = spark.sql("SELECT COUNT(*) as c FROM workspace.scada_cleaned.sensor_log").collect()[0]["c"]
    log_step("sensor_log", "SUCCESS", cnt)
except Exception as e:
    log_step("sensor_log", "FAILED", error_message=str(e))
    raise

In [0]:
try:
    spark.sql("""
        CREATE OR REPLACE TABLE workspace.scada_cleaned.alerts AS
        SELECT
          TRIM(IDD) AS alert_id, REPORT_DATE AS report_date,
          TRIM(ALERT_NAME) AS alert_type,
          TRY_CAST(ALERT_VALUE AS DOUBLE) AS alert_value,
          REPLACE(ALERT_RANGE, '&nbsp;', ' ') AS alert_range_raw,
          TRY_CAST(REGEXP_EXTRACT(REPLACE(ALERT_RANGE, '&nbsp;', ' '), 'Min:\\\\s*([\\\\d.]+)', 1) AS DOUBLE) AS range_min,
          TRY_CAST(REGEXP_EXTRACT(REPLACE(ALERT_RANGE, '&nbsp;', ' '), 'Max:\\\\s*([\\\\d.]+)', 1) AS DOUBLE) AS range_max,
          TRY_CAST(SERVER_DT AS TIMESTAMP) AS server_timestamp,
          TRY_CAST(UID AS INT) AS uid,
          TRIM(OTHER_2) AS parameter_name
        FROM workspace.scada_flow_2026.tb_report_alert
    """)
    cnt = spark.sql("SELECT COUNT(*) as c FROM workspace.scada_cleaned.alerts").collect()[0]["c"]
    log_step("alerts", "SUCCESS", cnt)
except Exception as e:
    log_step("alerts", "FAILED", error_message=str(e))
    raise

In [0]:
try:
    spark.sql("""
        CREATE OR REPLACE TABLE workspace.scada_cleaned.totalizer AS
        SELECT
          UID AS uid, REPORT_DATE AS report_date, TRIM(IMEI_NUMBER) AS imei,
          TRY_CAST(START_DAY AS DOUBLE) AS start_day_reading,
          TRY_CAST(END_DAY AS DOUBLE) AS end_day_reading,
          TRY_CAST(END_DAY AS DOUBLE) - TRY_CAST(START_DAY AS DOUBLE) AS daily_consumption,
          CASE
            WHEN UPPER(TRIM(PNAME)) LIKE '%TOTAL%MLD%' THEN 'Total MLD'
            WHEN UPPER(TRIM(PNAME)) LIKE '%TOTALIZER%M3%' THEN 'Totalizer (m3)'
            WHEN UPPER(TRIM(PNAME)) LIKE '%TOTAL%M3%' THEN 'Total (m3)'
            WHEN UPPER(TRIM(PNAME)) LIKE '%TOTAL%KL%' THEN 'Total (KL)'
            ELSE TRIM(PNAME)
          END AS parameter_name,
          TRIM(OTHER_3) AS unit, SERVER_DT AS server_timestamp
        FROM workspace.scada_flow_2026.tb_report_totalizer
        WHERE TRY_CAST(END_DAY AS DOUBLE) IS NOT NULL
    """)
    cnt = spark.sql("SELECT COUNT(*) as c FROM workspace.scada_cleaned.totalizer").collect()[0]["c"]
    log_step("totalizer", "SUCCESS", cnt)
except Exception as e:
    log_step("totalizer", "FAILED", error_message=str(e))
    raise

In [0]:
print("\n" + "="*60)
print("DATA QUALITY VALIDATION")
print("="*60)

validation_passed = True

# Check 1: Row counts (minimum thresholds)
thresholds = {
    "device_list": 500,
    "device_parameters": 15000,
    "sensor_log": 25000000,
    "alerts": 50000,
    "totalizer": 100000
}

for table, min_rows in thresholds.items():
    cnt = spark.sql(f"SELECT COUNT(*) as c FROM workspace.scada_cleaned.{table}").collect()[0]["c"]
    status = "PASS" if cnt >= min_rows else "WARN"
    if status == "WARN":
        validation_passed = False
    print(f"  [{status}] {table}: {cnt:,} rows (threshold: {min_rows:,})")

# Check 2: Freshness - latest sensor data should be within 48 hours of now
freshness = spark.sql("""
    SELECT
      MAX(data_timestamp) as latest,
      DATEDIFF(HOUR, MAX(data_timestamp), CURRENT_TIMESTAMP()) as hours_stale
    FROM workspace.scada_cleaned.sensor_log
""").collect()[0]
print(f"\n  Sensor data freshness: latest = {freshness['latest']}, {freshness['hours_stale']} hours old")

log_step("validation", "PASS" if validation_passed else "WARN")
print(f"\nValidation: {'PASSED' if validation_passed else 'WARNINGS detected'}")

In [0]:
run_end = datetime.now()
total_duration = (run_end - run_start).total_seconds()

print("\n" + "="*60)
print("PIPELINE RUN COMPLETE")
print("="*60)
print(f"  Run ID:    {run_id}")
print(f"  Started:   {run_start}")
print(f"  Finished:  {run_end}")
print(f"  Duration:  {total_duration:.1f}s ({total_duration/60:.1f} min)")

# Show run log
print("\nRun log:")
display(spark.sql(f"SELECT * FROM workspace.scada_pipeline.run_log WHERE run_id = '{run_id}' ORDER BY completed_at"))